# Сегментация изображений: попиксельная разметка снимков

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Сегментация изображений».

## Что делает метод

Сегментация изображений отвечает не на вопрос «что на снимке», а на вопрос «где именно это находится». Алгоритм размечает каждый пиксель: вот объект, вот фон, вот граница структуры. Для исследователя это снимает рутину ручной обводки объектов и даёт точные воспроизводимые измерения — площадь, форму, число структур.

Наиболее распространённая архитектура для научных задач — U-Net: кодировщик сжимает изображение и выделяет признаки, декодировщик восстанавливает попиксельную карту, а связи между ними сохраняют точные границы. В этом блокноте мы обучим компактную U-Net на синтетических снимках, оценим качество по коэффициенту Дайса и разберём результат. Блокнот рассчитан на запуск без графического ускорителя; время прохождения — около пяти минут.

## Интуиция за методом

Представьте, что вам нужно разметить карту города: не просто сказать «здесь есть парк», а аккуратно обвести каждый участок зелени, каждое здание, каждую дорогу. Именно это делает алгоритм сегментации — только не с картой, а с научным снимком, и не вручную, а автоматически для каждого из тысяч пикселей.

Алгоритм обучается на примерах: ему показывают снимок и эталонную разметку («маска» — изображение, где белый цвет означает «объект», чёрный — «фон»). После обучения он умеет по новому снимку самостоятельно нарисовать такую маску.

Архитектура U-Net работает как «испорченный телефон наоборот»: сначала изображение сжимается (кодировщик выделяет признаки, теряя детали положения), затем разворачивается обратно (декодировщик восстанавливает пространственные детали). Прямые связи между кодировщиком и декодировщиком — «шпаргалки» — передают точные координаты границ, не давая им потеряться при сжатии.

**Ключевые понятия, которые встретятся в блокноте:**
- **Маска (пиксельная маска)** — изображение того же размера, что и снимок; каждый пиксель в ней — 1 (объект) или 0 (фон).
- **Кодировщик** — часть сети, которая сжимает изображение и извлекает признаки (что нарисовано).
- **Декодировщик** — часть сети, которая разворачивает признаки обратно в пространственную карту (где именно).
- **Коэффициент Дайса** — мера совпадения двух масок: 0 означает полное несовпадение, 1 — идеальное; значения выше 0,85 обычно считаются хорошим результатом.
- **Функция потерь (BCEWithLogitsLoss)** — число, которое модель минимизирует при обучении; чем оно меньше, тем точнее маска.

## 1. Установка библиотек

In [ ]:
%pip install -q torch==2.3.1 numpy==1.26.4 matplotlib==3.9.2

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Сформируем синтетический набор микроскопических снимков: на сером фоне со случайным шумом разбросаны округлые светлые «клетки». Для каждого снимка известна точная маска — какие пиксели принадлежат клеткам. Такой набор воспроизводит структуру задачи биомедицинской сегментации и не требует загрузки внешних файлов и ручной разметки.

**Что делает ячейка ниже:** создаёт 200 пар «снимок + маска» и показывает несколько примеров, чтобы вы видели, с какими данными работает модель.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
IMG = 64                      # сторона снимка в пикселях


def make_sample():
    """Создаёт один снимок и точную маску объектов."""
    image = rng.normal(0.35, 0.06, (IMG, IMG)).astype("float32")
    mask = np.zeros((IMG, IMG), dtype="float32")
    yy, xx = np.mgrid[0:IMG, 0:IMG]
    for _ in range(rng.integers(3, 7)):
        cy, cx = rng.integers(10, IMG - 10, size=2)
        r = rng.integers(5, 10)
        disk = (yy - cy) ** 2 + (xx - cx) ** 2 <= r ** 2
        image[disk] += rng.uniform(0.4, 0.6)
        mask[disk] = 1.0
    return np.clip(image, 0, 1), mask


n_train, n_test = 160, 40
data = [make_sample() for _ in range(n_train + n_test)]
images = np.stack([d[0] for d in data])[:, None, :, :]
masks = np.stack([d[1] for d in data])[:, None, :, :]

X_train, X_test = images[:n_train], images[n_train:]
M_train, M_test = masks[:n_train], masks[n_train:]
print(f"Обучающих снимков: {len(X_train)}, тестовых: {len(X_test)}")
print(f"Размер снимка: {X_train.shape[2:]}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Показываем несколько примеров исходных снимков и их масок.
# Верхний ряд — снимки (яркие круги — «клетки»).
# Нижний ряд — эталонные маски (белое = объект, чёрное = фон).
n_preview = 5
fig, axes = plt.subplots(2, n_preview, figsize=(3.0 * n_preview, 6.2))
for j in range(n_preview):
    axes[0, j].imshow(images[j, 0], cmap="gray", vmin=0, vmax=1)
    axes[1, j].imshow(masks[j, 0], cmap="gray", vmin=0, vmax=1)
    for i in range(2):
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])
        axes[i, j].grid(False)
axes[0, 0].set_ylabel("Снимок", fontsize=12)
axes[1, 0].set_ylabel("Эталонная маска", fontsize=12)
fig.suptitle("Примеры обучающих данных: снимки и эталонные маски",
             fontsize=13)
fig.tight_layout()
plt.show()
print(f"Обучающих снимков: {len(X_train)}, тестовых: {len(X_test)}")
print(f"Размер снимка: {X_train.shape[2:]}")

**Как читать этот график.** Верхний ряд — сырые снимки: светлые округлые пятна это «клетки», серый шум — фон. Нижний ряд — соответствующие эталонные маски: белые области точно совпадают с клетками. Именно такие пары «снимок — маска» нужны для обучения: модель учится по снимку воспроизводить маску.

## 4. Применение метода

### Шаг 1. Архитектура U-Net

Соберём компактную U-Net. Ниже — краткая схема её работы:

```
Вход (снимок)
    → Кодировщик: слой 1 (16 фильтров) → слой 2 (32 фильтра)
         ↓ сжатие (MaxPool)                ↓ сжатие (MaxPool)
              → «Бутылочное горлышко» (64 фильтра)
         ↑ растяжение (ConvTranspose)      ↑ растяжение (ConvTranspose)
    ← Декодировщик: слой 2 + шпаргалка  ← слой 1 + шпаргалка
→ Выход: вероятность для каждого пикселя (1 канал)
```

«Шпаргалки» (skip connections, или прямые связи) — это прямые соединения от кодировщика к декодировщику на одном масштабе. Они передают точные координаты границ, которые иначе потерялись бы при сжатии.

**Что делает ячейка ниже:** определяет класс TinyUNet и создаёт экземпляр модели.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)


def conv_block(in_ch, out_ch):
    """Два свёрточных слоя с нелинейностью."""
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.ReLU(),
        nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.ReLU(),
    )


class TinyUNet(nn.Module):
    """Компактная U-Net для бинарной сегментации."""

    def __init__(self):
        super().__init__()
        self.enc1 = conv_block(1, 16)
        self.enc2 = conv_block(16, 32)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = conv_block(32, 64)
        self.up2 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec2 = conv_block(64, 32)
        self.up1 = nn.ConvTranspose2d(32, 16, 2, stride=2)
        self.dec1 = conv_block(32, 16)
        self.head = nn.Conv2d(16, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)                       # точные границы
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))
        d2 = self.dec2(torch.cat([self.up2(b), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)                    # логиты по пикселям


device = "cuda" if torch.cuda.is_available() else "cpu"
model = TinyUNet().to(device)
print(f"Устройство вычислений: {device}")

### Шаг 2. Обучение модели

**Что делает ячейка ниже:** запускает цикл обучения на 20 эпох. На каждой эпохе модель видит все обучающие снимки, вычисляет функцию потерь (насколько предсказанная маска расходится с эталоном) и корректирует свои параметры, чтобы ошибка уменьшилась.

- **Эпоха** — один проход через все обучающие снимки.
- **Функция потерь BCEWithLogitsLoss** — бинарная перекрёстная энтропия; она штрафует за каждый неправильно размеченный пиксель.
- **Adam** — алгоритм оптимизации; адаптивно подбирает скорость коррекции параметров.

Вывод: число потерь должно уменьшаться от эпохи к эпохе — это признак того, что модель обучается.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(M_train))
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


def dice_score(pred_mask, true_mask, eps=1e-6):
    """Коэффициент Дайса: мера перекрытия предсказанной и истинной масок."""
    inter = (pred_mask * true_mask).sum()
    return ((2 * inter + eps) /
            (pred_mask.sum() + true_mask.sum() + eps))


loss_history = []
n_epochs = 20
for epoch in range(1, n_epochs + 1):
    model.train()
    epoch_loss, seen = 0.0, 0
    for xb, mb in train_loader:
        xb, mb = xb.to(device), mb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), mb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(xb)
        seen += len(xb)
    loss_history.append(epoch_loss / seen)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Эпоха {epoch:2d}: функция потерь {loss_history[-1]:.4f}")

### Шаг 3. Оценка качества на тестовых снимках

**Что делает ячейка ниже:** прогоняет обученную модель через 40 тестовых снимков (которых она не видела при обучении), применяет порог 0,5 (вероятность > 0,5 → объект), вычисляет коэффициент Дайса для каждого снимка.

**Коэффициент Дайса** = 2 × (площадь пересечения масок) / (сумма площадей масок). Результат от 0 до 1: чем ближе к 1, тем точнее совпадение предсказанной маски с эталоном.

In [ ]:
# Оценка качества на тестовых снимках.
model.eval()
with torch.no_grad():
    logits = model(torch.from_numpy(X_test).to(device))
    prob = torch.sigmoid(logits).cpu().numpy()
pred_masks = (prob > 0.5).astype("float32")

dices = [float(dice_score(pred_masks[i], M_test[i]))
         for i in range(len(M_test))]
print(f"Средний коэффициент Дайса на тесте: {np.mean(dices):.3f}")

### Визуализация результата: «до и после»

Ниже — ключевой «ага»-момент блокнота. Для четырёх тестовых снимков мы покажем три состояния рядом:
1. Исходный снимок (то, что видит модель).
2. Эталонная маска (то, как правильно).
3. Предсказание модели (то, что получилось).

Обратите внимание: модель никогда не видела эти снимки при обучении. Число под каждым столбцом — коэффициент Дайса: насколько точно предсказание совпадает с эталоном.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

n_show = 4
fig, axes = plt.subplots(3, n_show, figsize=(3.1 * n_show, 9.5),
                         gridspec_kw={"hspace": 0.12, "wspace": 0.06})
row_titles = ["Снимок (вход)", "Эталонная маска", "Предсказание модели"]

for j in range(n_show):
    axes[0, j].imshow(X_test[j, 0], cmap="gray", vmin=0, vmax=1)
    axes[1, j].imshow(M_test[j, 0], cmap="gray", vmin=0, vmax=1)
    axes[2, j].imshow(pred_masks[j, 0], cmap="gray", vmin=0, vmax=1)
    dice_val = dices[j]
    # Цвет подписи: зелёный при хорошем результате, оранжевый — при слабом.
    color = VIZ["series"][1] if dice_val >= 0.80 else VIZ["series"][2]
    axes[2, j].set_xlabel(f"Дайс = {dice_val:.3f}", fontsize=12,
                          color=color, fontweight="bold")
    for i in range(3):
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])
        axes[i, j].grid(False)

for i, title in enumerate(row_titles):
    axes[i, 0].set_ylabel(title, fontsize=11, labelpad=6)

fig.suptitle(
    f"Сегментация на тестовых снимках  |  средний Дайс = {np.mean(dices):.3f}",
    fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

**Как читать этот график.** Каждый столбец — один тестовый снимок. Верхний ряд — вход (сырой снимок со случайным шумом). Средний ряд — эталонная маска, которую нарисовал человек (или в нашем случае — алгоритм генерации данных). Нижний ряд — то, что предсказала модель. Хороший результат: нижний ряд визуально неотличим от среднего; коэффициент Дайса выделен зелёным. Если модель пропустила клетку или добавила лишнюю — это видно сразу.

**Что делают ячейки ниже:** строят два вспомогательных графика — кривую обучения и гистограмму коэффициента Дайса по всем тестовым снимкам.

In [ ]:
# Кривая обучения и распределение качества по снимкам.
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.0))
axes[0].plot(range(1, n_epochs + 1), loss_history, marker="o",
             color=VIZ["series"][0])
axes[0].set_title("Кривая обучения")
axes[0].set_xlabel("Эпоха")
axes[0].set_ylabel("Функция потерь")

axes[1].hist(dices, bins=15, color=VIZ["series"][1],
             edgecolor=VIZ["background"])
axes[1].axvline(np.mean(dices), color=VIZ["series"][2], linestyle="--",
                label="Среднее")
axes[1].set_title("Качество сегментации по снимкам")
axes[1].set_xlabel("Коэффициент Дайса")
axes[1].set_ylabel("Число снимков")
axes[1].legend(loc="upper left")

fig.tight_layout()
plt.show()

**Как читать эти графики.**

- **Кривая обучения (слева):** ось X — номер эпохи, ось Y — значение функции потерь. Кривая должна убывать и выходить на плато. Если к последней эпохе она всё ещё круто падает — модель недообучена и выиграет от дополнительных эпох. Если в середине стала расти — переобучение.

- **Распределение Дайса (справа):** ось X — значение коэффициента Дайса (от 0 до 1), ось Y — число снимков с таким значением. Идеальный результат: все столбцы сдвинуты вправо, близко к 1. Длинный левый хвост (снимки с низким Дайсом) указывает на проблемные случаи — их стоит изучить отдельно. Вертикальная пунктирная линия — среднее.

## 5. Интерпретация результата

- **Коэффициент Дайса** меняется от 0 до 1 и измеряет перекрытие предсказанной и истинной масок; значения выше 0,8–0,9 обычно считаются хорошим результатом. Метрика accuracy здесь вводит в заблуждение, если объект занимает малую долю снимка.
- **Сравнение масок**: модель должна повторять не только расположение, но и форму и границы объектов. Систематические ошибки на границах указывают на нехватку обучающих данных или слишком грубую разметку.
- **Кривая обучения** должна снижаться и выходить на плато.
- **Распределение Дайса по снимкам**: длинный левый хвост означает, что часть снимков сегментируется заметно хуже — их стоит рассмотреть отдельно.

Важно: семантическая сегментация не разделяет слипшиеся объекты — для подсчёта отдельных структур нужна сегментация экземпляров. Качество следует проверять на снимках из независимого эксперимента.

## 5а. Поэкспериментируйте сами

Ниже — три простых изменения, которые дают наглядный результат. Измените параметр, перезапустите раздел 4 (ячейки обучения и оценки) и сравните.

| Что изменить | Где | Ожидаемый эффект |
|---|---|---|
| `n_epochs = 40` вместо 20 | ячейка обучения | Дайс вырастет; кривая потерь опустится ниже |
| `n_train = 40` вместо 160 | ячейка данных | Меньше данных → ниже Дайс, хуже маски |
| `r = rng.integers(3, 5)` вместо `(5, 10)` | `make_sample` | Клетки мельче → задача сложнее, проверьте границы |

## 6. Попробуйте на своих данных

Для своих снимков подготовьте два массива: изображения формы `(наблюдение, 1, высота, ширина)` и маски той же формы со значениями 0 и 1. Снимки приводят к единому размеру и нормируют яркость в диапазон [0, 1].

1. Загрузите изображения и маски в Colab через панель «Файлы».
2. Снимите комментарии в ячейке ниже и укажите пути к снимкам и маскам.
3. Если размеченных масок нет, рассмотрите готовые предобученные решения для своей предметной области (например, Cellpose для клеток или Segment Anything для объектов общего вида).
4. Последовательно выполните ячейки разделов 4 и 5.

In [ ]:
# --- Шаблон загрузки своих данных ---
# from pathlib import Path
# from PIL import Image
#
# size = 64                                     # сторона снимка после приведения
# img_dir = Path("my_images")                   # папка со снимками
# mask_dir = Path("my_masks")                   # папка с масками (имена совпадают)
#
# imgs, msks = [], []
# for path in sorted(img_dir.glob("*")):
#     img = Image.open(path).convert("L").resize((size, size))
#     msk = Image.open(mask_dir / path.name).convert("L").resize((size, size))
#     imgs.append(np.asarray(img, dtype="float32") / 255.0)
#     msks.append((np.asarray(msk, dtype="float32") > 127).astype("float32"))
# images = np.stack(imgs)[:, None, :, :]
# masks = np.stack(msks)[:, None, :, :]
#
# print(f"Снимков: {len(images)}")
# Далее разбейте на обучение и тест и повторите ячейки раздела 4.

## 7. Краткие выводы

- Сегментация — это попиксельная классификация: модель присваивает каждому пикселю метку «объект» или «фон».
- U-Net эффективна для научных снимков благодаря прямым связям (шпаргалкам), сохраняющим точные границы.
- Качество измеряется коэффициентом Дайса; значения выше 0,85 считаются хорошими для большинства задач.
- Метрика точности (accuracy) вводит в заблуждение, если объект занимает малую часть снимка, — всегда используйте Дайс или IoU.
- Семантическая сегментация не разделяет слипшиеся объекты; для подсчёта отдельных структур нужна сегментация экземпляров.
- Для реальных снимков рассмотрите готовые предобученные модели (Cellpose, Segment Anything) — они требуют значительно меньше разметки.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. На гистограмме коэффициента Дайса по тестовым снимкам большинство значений сосредоточено вблизи 0.9, но несколько снимков дали Дайс ниже 0.5. Какие два типа трудных случаев наиболее вероятно объясняют эти выбросы в задаче сегментации клеток?
2. Вы заменяете синтетические данные реальными микроскопическими снимками клеток, где клетки занимают лишь 3–5% площади снимка. Почему метрика accuracy (доля верно классифицированных пикселей) будет вводить в заблуждение, и какую метрику следует использовать вместо неё?
3. Вы обучили модель на снимках одного микроскопа и тестируете на снимках другого с иными настройками контраста. Коэффициент Дайса резко снизился. Назовите один технический приём предобработки и один приём аугментации, которые помогут снизить эту проблему.

<details>
<summary>Показать ориентиры для ответов</summary>

1. Наиболее вероятные причины: слипшиеся клетки (семантическая сегментация не разделяет касающиеся объекты, и маска «сливается»), а также клетки, частично попавшие на границу снимка — модель видела их меньше в обучении.
2. При сильном дисбалансе классов (97% фона) модель, всегда предсказывающая «фон», получит accuracy 0.97 при нулевой Дайс. Коэффициент Дайса (или IoU) явно учитывает оба класса и не зависит от дисбаланса.
3. Предобработка: нормализация гистограммы яркости (histogram equalization / CLAHE) приводит контраст разных микроскопов к сопоставимому диапазону. Аугментация: случайные изменения яркости и контраста при обучении (ColorJitter) заставляют модель быть устойчивой к вариациям освещения.
</details>
